# CleanVision — Model Training Notebook

Train the CNN cleanliness classifier on Google Colab (free GPU).

## Steps
1. Upload your dataset images to Google Drive
2. Mount Drive and point to the dataset
3. Train the model
4. Download the trained `cleanliness_model.h5` and place it in `backend/cleanliness_model.h5`

In [ ]:
#@title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title 2. Configuration — set your dataset path
DATASET_PATH = '/content/drive/MyDrive/cleanvision_dataset'  #@param {type:"string"}
# Expected structure:
#   cleanvision_dataset/
#     clean/       (hundreds of clean room images)
#     dirty/       (hundreds of dirty room images)

import os
print('Dataset contents:')
for d in ['clean', 'dirty']:
    p = os.path.join(DATASET_PATH, d)
    if os.path.exists(p):
        count = len(os.listdir(p))
        print(f'  {d}/: {count} images')
    else:
        print(f'  {d}/: NOT FOUND')

In [ ]:
#@title 3. Install dependencies
!pip install tensorflow pillow matplotlib

In [ ]:
#@title 4. Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
import os
import matplotlib.pyplot as plt

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
#@title 5. Create Datasets (Matching Backend Preprocessing)
# Backend uses: RGB, 224x224, /255.0 normalization.

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# Create dataset from directory
# label_mode 'binary' assigns 0 to 'clean' and 1 to 'dirty' alphabetically.
# This matches the backend: class_indices = {'clean': 0, 'dirty': 1}
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

print(f"Class names: {raw_train_ds.class_names}")

# Normalize by dividing by 255.0 to match backend exactly
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = raw_train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = raw_val_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
#@title 6. Define MobileNetV2 with Transfer Learning
# Load pre-trained MobileNetV2 without the top classification layer
base_model = MobileNetV2(input_shape=(224, 224, 3),
                         include_top=False,
                         weights='imagenet')

# Freeze the base model
base_model.trainable = False

# Create the new model on top
inputs = keras.Input(shape=(224, 224, 3))
# Inputs are already normalized
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
# Binary classification uses a single unit with a sigmoid activation
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)
model.summary()

In [ ]:
#@title 7. Compile and Train
EPOCHS = 10  #@param {type:"integer"}
LR = 0.001  #@param {type:"number"}

model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=['accuracy'])

# Train the model
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

In [ ]:
#@title 8. Plot Training Curves
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Training Loss', marker='o')
plt.plot(epochs_range, val_loss, label='Validation Loss', marker='s')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='s', color='green')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.grid(True)
plt.show()

In [ ]:
#@title 9. Save and Download Model
# Save the model in HDF5 format (.h5) exactly as backend expects
model_path = '/content/cleanliness_model.h5'
model.save(model_path)
print(f'Model saved to {model_path}')

from google.colab import files
print('Downloading cleanliness_model.h5...')
files.download(model_path)
print('\nSave this file as: backend/cleanliness_model.h5')